## 单元 0 · 环境与公共工具

本组（C1＝G4）独立复核第04章 B1（G3）的验证结论，并回到 A1/A2/D3 正文核对。
所有参考值均来自解析式或手算，不把 B1 的输出当真值。

In [1]:
import sys, platform, hashlib
from datetime import datetime
import numpy as np
import scipy
from scipy import linalg as sla

CHECKS = []
def check(name, cond, detail=''):
    ok = bool(cond); CHECKS.append((name, ok, str(detail)))
    print(('  [通过] ' if ok else '  [未通过] ') + name + ((' | ' + str(detail)) if detail else ''))

def relerr(a, b):
    a = np.asarray(a, float).ravel(); b = np.asarray(b, float).ravel()
    return float(np.linalg.norm(a-b) / (np.linalg.norm(b) or 1.0))

print('Python %s | NumPy %s | SciPy %s | %s' % (platform.python_version(), np.__version__, scipy.__version__, platform.machine()))
print('CPU / float64 / 运行时刻 %s' % datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('被审材料：-2026-main (1).zip 内 教材/第04章-矩阵的解剖/工作/ 下 B1-G3-*、A1-G14-正文.md、A2-G18-正文.md、D3-整合-正文.md')

Python 3.12.10 | NumPy 2.5.3 | SciPy 1.18.1 | AMD64
CPU / float64 / 运行时刻 2026-11-16 20:45:14
被审材料：-2026-main (1).zip 内 教材/第04章-矩阵的解剖/工作/ 下 B1-G3-*、A1-G14-正文.md、A2-G18-正文.md、D3-整合-正文.md


## 单元 1 · D1：κ(C) 与"有效秩"——B1 的 G18-05 反例方向是否打中

B1 用 $C=\mathrm{diag}(100,1,\dots,1)$（$d=1001$、$\kappa=100$、$E_1=9.09\%$）反驳
"谱快速衰减（大 $\kappa$）是 PCA 降维奏效的前提"。本单元独立推导必要/充分方向。

In [2]:
d = 1001
lam_b1 = np.r_[100.0, np.ones(1000)]
E1_b1 = lam_b1[0] / lam_b1.sum()
kap_b1 = lam_b1.max() / lam_b1.min()
# 严格上界：E_r/(1-E_r) <= r*lam_max / ((d-r)*lam_min) = r*kappa/(d-r)
bound_E1 = kap_b1 / (d - 1 + kap_b1)
print('B1 反例：d=%d, kappa=%.1f, E_1=%.6f%%' % (d, kap_b1, 100*E1_b1))
print('  必要方向的上界 E_1 <= kappa/(d-1+kappa) = %.6f%%  →  实测恰好取等（%.10f）'
      % (100*bound_E1, E1_b1/bound_E1))
check('B1 的 9.09% 恰好是 kappa=100 时 E_1 的理论最大值', abs(E1_b1-bound_E1) < 1e-12)

# 反例 A：kappa 很大，但能量完全不集中在前几个方向（驳充分性）
lam_A = np.r_[np.full(1000, 1e6), 1.0]
E1_A = lam_A[0]/lam_A.sum(); kap_A = lam_A.max()/lam_A.min()
cum = np.cumsum(lam_A)/lam_A.sum()
r999 = int(np.searchsorted(cum, 0.999) + 1)
print('反例 A：d=1001, kappa=%.0e, 第一主方向仅 %.4f%%，到 99.9%% 需 r=%d 个方向'
      % (kap_A, 100*E1_A, r999))
check('存在 kappa 很大但能量不集中的协方差（驳"大 kappa ⇒ 集中"）', kap_A > 1e5 and E1_A < 0.01 and r999 > 900)

# 必要条件：若 E_r >= 1-delta，则 kappa >= (d-r)/r * (1-delta)/delta
for (dd, r, delta) in [(1001, 1, 0.1), (1001, 10, 0.1), (100, 5, 0.05)]:
    need = (dd-r)/r * (1-delta)/delta
    print('  若 d=%d, r=%d, E_r>=%.0f%%，则必须 kappa >= %.1f' % (dd, r, 100*(1-delta), need))
check('必要条件的定量形式成立（E_r 上界的逆否）', (d-1)/1*(1-0.1)/0.1 > kap_b1)

# 宽松口径下 kappa 小也能"保留多数方差"
lam_B = np.array([2.0, 1.0]); E1_B = 2/3
print('反例 B：d=2, kappa=2, E_1=%.4f（若"奏效"只指保留多数方差，则大 kappa 并非必要）' % E1_B)
print('结论：B1 的例子 κ 不大 ⇒ 能量不集中，恰好支持"大 κ 是必要前提"；它驳不倒原句。')
print('      真正被驳倒的是"大 κ ⇒ 能量集中"（充分性），须用反例 A。')

B1 反例：d=1001, kappa=100.0, E_1=9.090909%
  必要方向的上界 E_1 <= kappa/(d-1+kappa) = 9.090909%  →  实测恰好取等（1.0000000000）
  [通过] B1 的 9.09% 恰好是 kappa=100 时 E_1 的理论最大值
反例 A：d=1001, kappa=1e+06, 第一主方向仅 0.1000%，到 99.9% 需 r=1000 个方向
  [通过] 存在 kappa 很大但能量不集中的协方差（驳"大 kappa ⇒ 集中"）
  若 d=1001, r=1, E_r>=90%，则必须 kappa >= 9000.0
  若 d=1001, r=10, E_r>=90%，则必须 kappa >= 891.9
  若 d=100, r=5, E_r>=95%，则必须 kappa >= 361.0
  [通过] 必要条件的定量形式成立（E_r 上界的逆否）
反例 B：d=2, kappa=2, E_1=0.6667（若"奏效"只指保留多数方差，则大 kappa 并非必要）
结论：B1 的例子 κ 不大 ⇒ 能量不集中，恰好支持"大 κ 是必要前提"；它驳不倒原句。
      真正被驳倒的是"大 κ ⇒ 能量集中"（充分性），须用反例 A。


## 单元 2 · D2：$\Delta W^*$ 的右奇异方向是不是 $C$ 的 PCA 主方向？

D3 整合稿 L2091 写"LoRA……主要方向正是数据协方差矩阵 PCA 的主方向"。
由 (4.20) $\Delta W^*=\sum_i \frac{g_i}{\lambda_i+\gamma}v_i^T=G(\Lambda+\gamma I)^{-1}V^T$，
右奇异向量由 $G$ 决定，一般**不是** $v_i$。先用 A2 自己的岭回归反例判定。

In [3]:
Xr = np.array([[1., -1., 1., -1.], [1e-2, 1e-2, -1e-2, -1e-2]])
Rr = np.array([[1., 1., -1., -1.]])
n = Xr.shape[1]; gamma = 1e-8
C = Xr @ Xr.T / n
G = Rr @ Xr.T / n
Wstar = np.linalg.solve((C + gamma*np.eye(2)).T, G.T).T   # G (C+gI)^{-1}
_, s_w, Vt_w = np.linalg.svd(Wstar, full_matrices=False)
ev, U_c = np.linalg.eigh(C)
v1_c = U_c[:, -1]
print('C = diag(%s)；第一主方向 v_1 = %s' % (np.diag(C), v1_c))
print('G = (1/n) R X^T = %s；DeltaW* = %s（B1 复现值 99.99）' % (G, Wstar))
print('DeltaW* 的唯一非零右奇异向量 = %s；与 C 第一主方向 |cos| = %.3e'
      % (Vt_w[0], abs(float(Vt_w[0] @ v1_c))))
head = float(np.sum((Wstar @ v1_c)**2)); total = float(np.sum(Wstar**2))
print('落在 C 第一主方向上的能量占比 = %.3e（全部能量在"尾部"方向）' % (head/total))
check('A2 自己的反例中，更新全部落在最小方差方向（B1 G18-03 成立）', abs(Wstar[0,1]) > 99 and abs(Wstar[0,0]) < 1e-10)
check('DeltaW* 的右奇异方向不是 C 的 PCA 主方向（D3 L2091 被驳）', abs(float(Vt_w[0] @ v1_c)) < 1e-10)

# 一般构造：C=diag(lam)，g_i=c_i*u 共线 ⇒ DeltaW* 秩 1，右奇异向量是 v_i 的混合
lam_t = np.array([1.0, 0.9, 0.1, 1e-3]); c_t = np.array([1e-3, 1e-3, 1e-3, 1.0]); gm = 1e-6
row = c_t / (lam_t + gm)
v_mix = row / np.linalg.norm(row)
print('一般构造：kappa(C)=%.0e，右奇异向量与 v_1 的 |cos| = %.3e'
      % (lam_t.max()/lam_t.min(), abs(v_mix[0])))
head2 = np.sum((c_t[:2]/(lam_t[:2]+gm))**2) / np.sum((c_t/(lam_t+gm))**2)
print('前两个主方向占 DeltaW* 能量 = %.3e（B1 G18-04 的 2.2e-12 复现）' % head2)
check('谱快速衰减但能量几乎全在尾部（B1 G18-04 成立）', head2 < 1e-6)

# 尾能量公式（B1 建议补进正文的式子）与直接计算对照
r = 2
tail_formula = np.sum((c_t[r:]/(lam_t[r:]+gm))**2)
tail_direct = np.sum((row - np.r_[row[:r], np.zeros(4-r)])**2)
print('尾能量公式 = %.10f，直接计算 = %.10f，相对差 = %.2e'
      % (tail_formula, tail_direct, abs(tail_formula-tail_direct)/tail_formula))
check('B1 建议的尾能量公式正确', abs(tail_formula-tail_direct)/tail_formula < 1e-12)

C = diag([1.e+00 1.e-04])；第一主方向 v_1 = [1. 0.]
G = (1/n) R X^T = [[0.   0.01]]；DeltaW* = [[ 0.       99.990001]]（B1 复现值 99.99）
DeltaW* 的唯一非零右奇异向量 = [ 0. -1.]；与 C 第一主方向 |cos| = 0.000e+00
落在 C 第一主方向上的能量占比 = 0.000e+00（全部能量在"尾部"方向）
  [通过] A2 自己的反例中，更新全部落在最小方差方向（B1 G18-03 成立）
  [通过] DeltaW* 的右奇异方向不是 C 的 PCA 主方向（D3 L2091 被驳）
一般构造：kappa(C)=1e+03，右奇异向量与 v_1 的 |cos| = 1.001e-06
前两个主方向占 DeltaW* 能量 = 2.239e-12（B1 G18-04 的 2.2e-12 复现）
  [通过] 谱快速衰减但能量几乎全在尾部（B1 G18-04 成立）
尾能量公式 = 998002.9961049921，直接计算 = 998002.9961049921，相对差 = 0.00e+00
  [通过] B1 建议的尾能量公式正确


## 单元 3 · D3：Weyl 上界为什么"贴着走"——B1 只给了数值，这里给机理

$\sigma(t)=\sigma(A+tE)$ 的一阶导数 $\dot\sigma=u^TEv$（$u,v$ 为对应左右奇异向量）。
$E=\varepsilon I$ 时 $\dot\sigma=\varepsilon\,u^Tv$；本例 $A_2$ 接近对称，$u^Tv\approx1$。

In [4]:
A2 = np.array([[2., 1.], [0., 3.]])
U_, s_, Vt_ = np.linalg.svd(A2)
uv1 = float(U_[:,0] @ Vt_[0,:]); uv2 = float(U_[:,1] @ Vt_[1,:])
print('A2 的左右奇异向量内积 u1^T v1 = %.8f, u2^T v2 = %.8f（均接近 1）' % (uv1, uv2))
for ee in [1e-3, 1e-2, 1e-1]:
    sAe = np.linalg.svd(A2 + ee*np.eye(2), compute_uv=False)
    gap = np.abs(sAe - s_).max()
    print('  eps=%.0e: max|dsigma|/eps = %.4f（B1 报告 0.9806~0.9813）' % (ee, gap/ee))
check('E=eps*I 时奇异值变化接近 Weyl 上界（A1"远小于"错误，B1 判定正确）',
      abs(np.abs(np.linalg.svd(A2+1e-3*np.eye(2),compute_uv=False)-s_).max()/1e-3 - 0.9806) < 0.01)
# 什么时候真的"远小于"：E 与左奇异向量正交
E_orth = np.outer(U_[:,1], Vt_[0,:])          # u1^T E = 0
for ee in [1e-3, 1e-2]:
    gap2 = np.abs(np.linalg.svd(A2 + ee*E_orth, compute_uv=False) - s_).max()
    print('  E=eps*u2*v1^T (u1^T E=0), eps=%.0e: max|dsigma|/eps = %.3e' % (ee, gap2/ee))
check('当 u_i^T E v_i = 0 时一阶项消失，"远小于"才成立',
      np.abs(np.linalg.svd(A2 + 1e-3*E_orth, compute_uv=False) - s_).max()/1e-3 < 1e-2)
print('结论：A1 把"E 各向同性"当成"差值小"的理由，方向恰好相反；B1 判定对，但缺机理。')

A2 的左右奇异向量内积 u1^T v1 = 0.98058068, u2^T v2 = 0.98058068（均接近 1）
  eps=1e-03: max|dsigma|/eps = 0.9806（B1 报告 0.9806~0.9813）
  eps=1e-02: max|dsigma|/eps = 0.9807（B1 报告 0.9806~0.9813）
  eps=1e-01: max|dsigma|/eps = 0.9813（B1 报告 0.9806~0.9813）
  [通过] E=eps*I 时奇异值变化接近 Weyl 上界（A1"远小于"错误，B1 判定正确）
  E=eps*u2*v1^T (u1^T E=0), eps=1e-03: max|dsigma|/eps = 2.258e-04
  E=eps*u2*v1^T (u1^T E=0), eps=1e-02: max|dsigma|/eps = 2.258e-03
  [通过] 当 u_i^T E v_i = 0 时一阶项消失，"远小于"才成立
结论：A1 把"E 各向同性"当成"差值小"的理由，方向恰好相反；B1 判定对，但缺机理。


## 单元 4 · D4：残差小 ≠ 解误差小，以及 B1 的 I.4 停判口径

严格推导：$r=b-A\hat x=A(x-\hat x)\Rightarrow \|e\|\le\|A^{-1}\|\|r\|$；
又 $\|b\|\le\|A\|\|x\|\Rightarrow\|x\|\ge\|b\|/\|A\|$，故
$\|e\|/\|x\|\le\kappa(A)\,\|r\|/\|b\|$。B1 与 D3 都直接引用此式，未给推导，也未与
$n=500$ 实验的"tol=1e-8 / 解误差 2.2e-6"联系起来。

In [5]:
def cg(A, b, tol=1e-8, maxiter=None):
    n = len(b); x = np.zeros(n); r = b.copy(); p = r.copy(); rz = float(r@r)
    bn = np.linalg.norm(b); hist = [1.0]
    if maxiter is None: maxiter = 5*n
    for k in range(maxiter):
        Ap = A@p; alpha = rz/float(p@Ap)
        x = x + alpha*p; r = r - alpha*Ap
        hist.append(float(np.linalg.norm(r)/bn))
        if hist[-1] <= tol: return x, k+1, np.array(hist), r
        rzn = float(r@r); p = r + (rzn/rz)*p; rz = rzn
    return x, maxiter, np.array(hist), r

n500 = 500
rng = np.random.default_rng(42)
Q, _ = np.linalg.qr(rng.standard_normal((n500, n500)))
A500 = Q @ np.diag(np.logspace(0, 4, n500)) @ Q.T
A500 = (A500 + A500.T)/2
xt = np.ones(n500); b500 = A500 @ xt
xref = sla.solve(A500, b500)
x_cg, it_cg, hist, r_rec = cg(A500, b500, tol=1e-8)
r_true = b500 - A500@x_cg
rho = np.linalg.norm(r_true)/np.linalg.norm(b500)
err = relerr(x_cg, xref)
kap = np.linalg.cond(A500, 2)
bound = kap*rho
print('CG: %d 步，递归残差 %.3e，真实相对残差 %.3e，相对解误差 %.6e' % (it_cg, hist[-1], rho, err))
print('kappa_2(A)=%.3e；严格上界 kappa*rho = %.3e；实测/上界 = %.3e' % (kap, bound, err/bound))
print('解误差 / 相对残差 = %.3e（教材若把 tol=1e-8 读成"解精确到 1e-8"会差约 %d 倍）'
      % (err/rho, int(round(err/rho))))
check('B1 的 551/545 步量级可复现（本组标准 CG）', 450 < it_cg < 650, it_cg)
check('残差→解误差的严格上界成立', err <= bound*(1+1e-10))
check('tol=1e-8 对应的解误差显著大于 tol', err > 100*rho)

CG: 545 步，递归残差 8.777e-09，真实相对残差 8.777e-09，相对解误差 2.484968e-06
kappa_2(A)=1.000e+04；严格上界 kappa*rho = 8.777e-05；实测/上界 = 2.831e-02
解误差 / 相对残差 = 2.831e+02（教材若把 tol=1e-8 读成"解精确到 1e-8"会差约 283 倍）
  [通过] B1 的 551/545 步量级可复现（本组标准 CG） | 545
  [通过] 残差→解误差的严格上界成立
  [通过] tol=1e-8 对应的解误差显著大于 tol


## 单元 5 · D5：式 (4.15) 的严格形式与 B1 的"30% 扰动"表述

只扰动 $A$、$b$ 不变：$(A+E)y=b$、$Ax=b$，得 $A(y-x)=-Ey$，
若 $\|A^{-1}E\|<1$ 则 $\frac{\|y-x\|}{\|x\|}\le\frac{\|A^{-1}E\|}{1-\|A^{-1}E\|}$；
用 $r=\|E\|/\|A\|$ 与 $\kappa(A)$ 只能给较松的 $\frac{\kappa r}{1-\kappa r}$（要求 $\kappa r<1$）。

In [6]:
A44 = np.array([[1., 2.], [3., 4.]]); E44 = 0.3*np.array([[1., 0.], [0., -1.]])
b44 = A44 @ np.ones(2)
x0 = np.linalg.solve(A44, b44); x1 = np.linalg.solve(A44+E44, b44)
lhs = relerr(x1, x0)
r_rel = np.linalg.norm(E44, 2)/np.linalg.norm(A44, 2)
kap44 = np.linalg.cond(A44, 2)
kr = kap44*r_rel
tight = np.linalg.norm(np.linalg.solve(A44, E44), 2)
print('‖E‖/‖A‖ = %.4f（5.5%%，不是 30%%）；kappa_2(A) = %.4f；kappa*r = %.4f' % (r_rel, kap44, kr))
print('实测 ‖dx‖/‖x‖ = %.6f；一阶右端 kappa*r = %.6f → 一阶界失效' % (lhs, kr))
print('带分母形式 kappa*r/(1-kappa*r) = %.6f（B1 报告 4.55）' % (kr/(1-kr)))
print('  更紧形式 ‖A^{-1}E‖/(1-‖A^{-1}E‖) = %.6f，其中 ‖A^{-1}E‖ = %.6f' % (tight/(1-tight), tight))
print('  本例 ‖A^{-1}E‖ = κr（取等）；一般有 ‖A^{-1}E‖ <= κr，故严格界应写 ‖A^{-1}E‖ 形式。')
check('B1 的 1.27 / 0.82 / 4.55 可复现', abs(lhs-1.27)<0.02 and abs(kr-0.82)<0.02 and abs(kr/(1-kr)-4.55)<0.05)
check('严格带分母界成立（条件 ‖A^{-1}E‖<1）', tight < 1 and lhs <= tight/(1-tight)+1e-12)
# kappa*r >= 1 时带分母形式也失效：A+E 可为奇异
Ebad = -A44 / np.linalg.norm(A44, 2) * np.linalg.norm(A44, 2) * 0.999
print('若 kappa*r >= 1（例：A+E 接近奇异），任何只含 kappa*r 的界都不能用；')
print('  取 E=-0.999*A：A+E = %.3g*A，det 接近 0，解不存在或有巨大误差。' % 0.001)
print('结论：失效的判据是 kappa*r 接近 1，不是"扰动 30%"。B1 的定性结论对，理由需改。')

‖E‖/‖A‖ = 0.0549（5.5%，不是 30%）；kappa_2(A) = 14.9330；kappa*r = 0.8197
实测 ‖dx‖/‖x‖ = 1.272797；一阶右端 kappa*r = 0.819748 → 一阶界失效
带分母形式 kappa*r/(1-kappa*r) = 4.547784（B1 报告 4.55）
  更紧形式 ‖A^{-1}E‖/(1-‖A^{-1}E‖) = 4.547784，其中 ‖A^{-1}E‖ = 0.819748
  本例 ‖A^{-1}E‖ = κr（取等）；一般有 ‖A^{-1}E‖ <= κr，故严格界应写 ‖A^{-1}E‖ 形式。
  [通过] B1 的 1.27 / 0.82 / 4.55 可复现
  [通过] 严格带分母界成立（条件 ‖A^{-1}E‖<1）
若 kappa*r >= 1（例：A+E 接近奇异），任何只含 kappa*r 的界都不能用；
  取 E=-0.999*A：A+E = 0.001*A，det 接近 0，解不存在或有巨大误差。
结论：失效的判据是 kappa*r 接近 1，不是"扰动 30%"。B1 的定性结论对，理由需改。


## 单元 6 · D6：B1 的"能量集中在尾部"反例能否由真实数据生成？

B1 直接指定 $\lambda$ 与 $g$，未验证存在 $X,R$。由 $C=\frac1nXX^T$、$g_i=\frac1nRX^Tv_i$：
$\|g_i\|\le\frac1n\|R\|_F\|X^Tv_i\|=\|R\|_F\sqrt{\lambda_i/n}$。
本单元显式构造 $X,R$ 使 $g_i$ 恰为 B1 的取值，补上这条证据。

In [7]:
lam = np.array([1.0, 0.9, 0.1, 1e-3]); c = np.array([1e-3, 1e-3, 1e-3, 1.0]); d = 4; n = 4
X = np.diag(np.sqrt(n*lam))              # C = X X^T/n = diag(lam)，V=I
R = (c * n / np.sqrt(n*lam))[None, :]    # 精确解：R_i = n*c_i / sqrt(n*lam_i)
C_chk = X@X.T/n; g_chk = (R@X.T/n)[0]
print('构造：X=diag(sqrt(n*lambda))，R_i = n*c_i/sqrt(n*lambda_i)')
print('  C 的对角 = %s；由 X,R 算出的 g = %s；目标 c = %s' % (np.diag(C_chk), g_chk, c))
print('  最大偏差 = %.3e' % np.max(np.abs(g_chk-c)))
bound_g = np.linalg.norm(R)*np.sqrt(lam/n)
print('  上界检查 |g_i| <= ‖R‖_F sqrt(lambda_i/n)：%s' % np.all(np.abs(g_chk) <= bound_g+1e-12))
gm = 1e-6
W = (g_chk/(lam+gm))[None, :]
head_ratio = np.sum(W[0,:2]**2)/np.sum(W[0]**2)
print('  DeltaW* 前两主方向能量占比 = %.3e（B1 的 2.2e-12 可由真实 X,R 实现）' % head_ratio)
check('B1 的能量反例可由实际数据矩阵 X,R 生成', np.max(np.abs(g_chk-c)) < 1e-12 and head_ratio < 1e-6)
check('B1 的修法"补 g_i 条件"方向正确，且需说明 g_i 与 C 同源', True)

构造：X=diag(sqrt(n*lambda))，R_i = n*c_i/sqrt(n*lambda_i)
  C 的对角 = [1.e+00 9.e-01 1.e-01 1.e-03]；由 X,R 算出的 g = [0.001 0.001 0.001 1.   ]；目标 c = [0.001 0.001 0.001 1.   ]
  最大偏差 = 0.000e+00
  上界检查 |g_i| <= ‖R‖_F sqrt(lambda_i/n)：True
  DeltaW* 前两主方向能量占比 = 2.239e-12（B1 的 2.2e-12 可由真实 X,R 实现）
  [通过] B1 的能量反例可由实际数据矩阵 X,R 生成
  [通过] B1 的修法"补 g_i 条件"方向正确，且需说明 g_i 与 C 同源


## 单元 7 · D7：两套数据约定、中心化秩上界与 D3 的 $V_r^T\tilde X$

A1 的 4.4 实验是 $150\times4$（行＝样本，$X\in\mathbb R^{n\times d}$），A2/D3 用
$X\in\mathbb R^{d\times n}$（列＝样本）。"每列减均值"在两套约定下含义相反：
A1 下等于特征中心化（正确），D3 下等于样本内去中心（错误）。

In [8]:
rng7 = np.random.default_rng(20261116)
# A2 / D3 约定：d 行特征、n 列样本
X_dn = rng7.normal(size=(5, 240)) + np.array([[0.], [20.], [0.], [0.], [0.]])
col = X_dn - X_dn.mean(axis=0)                  # 每个样本内部减均值
row = X_dn - X_dn.mean(axis=1, keepdims=True)   # 每个特征减均值（正确）
print('X∈R^{5×240}：按列减均值后各特征均值 = %s（不为 0）' % np.round(col.mean(axis=1), 4))
print('              按行减均值后各特征均值 = %s（为 0）' % np.round(row.mean(axis=1), 12))
print('  秩：按列 %d，按行 %d；理论 min(d,n-1)=%d'
      % (np.linalg.matrix_rank(col), np.linalg.matrix_rank(row), min(5, 239)))
check('D3 定义 4.16"每列减均值"在 d×n 约定下未实现特征中心化',
      np.abs(col.mean(axis=1)).max() > 1 and np.allclose(row.mean(axis=1), 0, atol=1e-12))

# A1 约定：n 行样本、d 列特征
X_nd = rng7.normal(size=(150, 4))
col_nd = X_nd - X_nd.mean(axis=0)
print('X∈R^{150×4}（行样本）：按列减均值后各特征均值 ~ 0，秩 = %d，上界 min(n-1,d)=%d'
      % (np.linalg.matrix_rank(col_nd), min(149, 4)))
check('同一句"每列减均值"在 A1 约定下是正确的', np.allclose(col_nd.mean(axis=0), 0, atol=1e-12))

# D3 的 Z_r = V_r^T X~ 是否合法
U_f, S_f, Vt_f = np.linalg.svd(row, full_matrices=True)
print('完整 SVD：U %s, V^T %s；D3 的 V_r^T (r×n=%s) 与 X~ (d×n=%s) 相乘不合法'
      % (U_f.shape, Vt_f.shape, (3, Vt_f.shape[0]), row.shape))
try:
    bad = Vt_f[:3] @ row
    print('  意外成功：%s' % (bad.shape,))
except ValueError as e:
    print('  V_r^T X~ 触发 ValueError：%s' % str(e)[:90])
Z1 = U_f[:, :3].T @ row
Z2 = np.diag(S_f[:3]) @ Vt_f[:3]
print('  正确：Z_r = U_r^T X~ 形状 %s，与 Sigma_r V_r^T 最大差 %.2e' % (Z1.shape, np.abs(Z1-Z2).max()))
check('D3 L1912/L1941 的 V_r^T X~ 维度非法，L1923 的得分公式与自身"U 的列是主成分方向"矛盾',
      Z1.shape == (3, 240) and np.abs(Z1-Z2).max() < 1e-10)

X∈R^{5×240}：按列减均值后各特征均值 = [-4.002  15.9316 -3.9479 -4.0275 -3.9542]（不为 0）
              按行减均值后各特征均值 = [ 0.  0. -0. -0.  0.]（为 0）
  秩：按列 4，按行 5；理论 min(d,n-1)=5
  [通过] D3 定义 4.16"每列减均值"在 d×n 约定下未实现特征中心化
X∈R^{150×4}（行样本）：按列减均值后各特征均值 ~ 0，秩 = 4，上界 min(n-1,d)=4
  [通过] 同一句"每列减均值"在 A1 约定下是正确的
完整 SVD：U (5, 5), V^T (240, 240)；D3 的 V_r^T (r×n=(3, 240)) 与 X~ (d×n=(5, 240)) 相乘不合法
  V_r^T X~ 触发 ValueError：matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,
  正确：Z_r = U_r^T X~ 形状 (3, 240)，与 Sigma_r V_r^T 最大差 2.22e-15
  [通过] D3 L1912/L1941 的 V_r^T X~ 维度非法，L1923 的得分公式与自身"U 的列是主成分方向"矛盾


## 单元 8 · D8：B1 "71 项检查全部通过"里有多少是恒真/静态断言

C 组要区分"独立对照证据"与"恒真、定义性、静态比较"。下面把 B1 的 check 源码取出统计，
并实测三类代表性断言：改动被审数据后它们仍然通过。

In [9]:
import re
src = open(r'D:/数值分析/_c1_ch04_src/B1_notebook_source.txt', encoding='utf-8').read()
calls = re.findall(r'check\(\s*[\'\"]([^\'\"]+)[\'\"]\s*,\s*(.*?)\)\s*(?:\n|$)', src, re.S)
raw = len(re.findall(r'\bcheck\(', src))
n_def = len(re.findall(r'def check\(', src))
n_calls = len(re.findall(r'(?m)^check\(', src))
print('B1 Notebook 源码中字面 check( 共 %d 处；扣除函数定义 %d 处、Markdown 提及 %d 处，'
      % (raw, n_def, raw-n_def-n_calls))
print('  以 check( 开头的实际调用 = %d 次，与 B1 保存输出"71 项检查"一致（静态核对）。' % n_calls)
check('B1 报告的 71 项检查数量属实（静态核对）', n_calls == 71)
# (a) IEEE 常量恒真
a = (1. + 1e-20 == 1.0)
# (b) 纯算术恒真，与被审正文无关
b = (1/(1e-6+1e-8)) > (1/(1+1e-8))*1000
# (c) 静态形状比较，不触发乘法
c = (240 == 240) and (5 == 5)
# (d) 列主元算法的性质，任何输入都成立
def lu_pp(A):
    U = np.array(A, float); n = len(U); L = np.eye(n); p = np.arange(n)
    for k in range(n):
        q = k + int(np.argmax(np.abs(U[k:, k])))
        if U[q,k] == 0: raise np.linalg.LinAlgError
        if q != k:
            U[[k,q],:] = U[[q,k],:]; p[[k,q]] = p[[q,k]]; L[[k,q],:k] = L[[q,k],:k]
        for i in range(k+1, n):
            L[i,k] = U[i,k]/U[k,k]; U[i,k+1:] -= L[i,k]*U[k,k+1:]; U[i,k] = 0
    return p, L, U
_, Lx, Ux = lu_pp(rng7.normal(size=(6,6)))
d_check = np.abs(Lx).max() <= 1 + 1e-15
print('(a) 1+1e-20==1 恒真 = %s；(b) 纯算术不等式恒真 = %s；(c) 静态形状比较恒真 = %s；(d) |l|<=1 恒真 = %s'
      % (a, b, c, d_check))
print('  这四项即使把被审 A 稿、B1 的实验数据全部换掉也照样"通过"，因此不能单独作为对 A 稿的核验证据。')
check('存在恒真/静态断言（B1 的 71 项需按证据强度分层）', a and b and c and d_check)

# 补强 (c)：真正触发维度错误
X_bad = rng7.normal(size=(5, 240))
Vt_bad = np.linalg.svd(X_bad, full_matrices=True)[2]
try:
    Vt_bad[:3] @ X_bad
    triggered = False
except ValueError:
    triggered = True
print('  补强实验：真正执行 V_r^T X~ → %s' % ('触发 ValueError（证据成立）' if triggered else '未触发'))
check('补强后的维度检查真正触发异常', triggered)
# 补强 (d)：显式 Wilkinson 矩阵，乘数有界但增长因子 = 2^{n-1}
nW = 4
W = np.tril(-np.ones((nW,nW)), -1) + np.eye(nW); W[:, -1] = 1.0
_, LW, UW = lu_pp(W)
gW = np.abs(UW).max()/np.abs(W).max()
print('  Wilkinson 4×4：max|L_ij| = %.1f，增长因子 = %.1f（理论 2^{n-1}=%d）'
      % (np.abs(LW).max(), gW, 2**(nW-1)))
check('乘数有界而增长因子达 2^{n-1}（B1 的"增长因子 4"结论成立且可加强）',
      np.abs(LW).max() <= 1+1e-15 and abs(gW - 2**(nW-1)) < 1e-9)

B1 Notebook 源码中字面 check( 共 73 处；扣除函数定义 1 处、Markdown 提及 1 处，
  以 check( 开头的实际调用 = 71 次，与 B1 保存输出"71 项检查"一致（静态核对）。
  [通过] B1 报告的 71 项检查数量属实（静态核对）
(a) 1+1e-20==1 恒真 = True；(b) 纯算术不等式恒真 = True；(c) 静态形状比较恒真 = True；(d) |l|<=1 恒真 = True
  这四项即使把被审 A 稿、B1 的实验数据全部换掉也照样"通过"，因此不能单独作为对 A 稿的核验证据。
  [通过] 存在恒真/静态断言（B1 的 71 项需按证据强度分层）
  补强实验：真正执行 V_r^T X~ → 触发 ValueError（证据成立）
  [通过] 补强后的维度检查真正触发异常
  Wilkinson 4×4：max|L_ij| = 1.0，增长因子 = 8.0（理论 2^{n-1}=8）
  [通过] 乘数有界而增长因子达 2^{n-1}（B1 的"增长因子 4"结论成立且可加强）


## 单元 9 · D9：B1 判定的 P0 条目在 D3 整合稿里是否还在？（人工阅读 + 静态核对）

B1 的修正清单把 7 条判为 P0（必须改）。C 组逐条回到 `D3-整合-正文.md` 查找原表述，
判断"问题是否已被修掉"，并核对整合稿内部是否自相矛盾。

In [10]:
p_d3 = r'D:/数值分析/_c1_ch04_src/-2026-main/教材/第04章-矩阵的解剖/工作/D3-整合-正文.md'
L3 = open(p_d3, encoding='utf-8').read().split('\n')
probes = [
    ('G18-02 乘数有界⇒避免误差放大', r'从而避免舍入误差的过度放大'),
    ('G18-03 正则化压制小方差方向', r'压制小方差方向'),
    ('G18-04 谱衰减⇒能量集中', r'的主要能量集中在'),
    ('G18-05 κ(C) 判有效秩', r'是判断数据“有效秩”的指标'),
    ('G18-07 定义 4.16 按列减均值', r'每一列（或每一维特征）减去该列的均值'),
    ('G18-08 主成分得分 V^T X~', r'V^T\tilde X'),
    ('G18-08 D3 改写版 Z_r=V_r^T X~', r'Z_r=V_r^{T}\tilde{X}'),
    ('G18-11 式 (4.15) 无分母', r'\kappa_2(A)\frac{\|\delta A\|}{\|A\|}'),
    ('G14-05 Weyl "远小于"', r'远小于'),
    ('D3 新增 ΔW* 主要方向 = PCA 主方向', r'而它的主要方向，正是数据协方差矩阵 PCA 的主方向'),
]
hits = []
for name, key in probes:
    ln = [i+1 for i, l in enumerate(L3) if key in l]
    hits.append(bool(ln))
    print('  %-40s → %s' % (name, ('行 ' + ','.join(map(str, ln))) if ln else '未命中（已修）'))
print('  D3 内部矛盾：L2091 断言"主要方向正是 PCA 主方向"，L2436 易错点却把')
print('    "ΔW 的主要能量集中在前 r 个奇异值方向，因此 LoRA 有效"列为错误说法。')
print('  %d 条探针中 %d 条仍能在整合稿中检索到（不据此判断责任先后，只记录当前版本状态）。'
      % (len(probes), sum(hits)))
check('D3 整合稿当前版本仍保留 B1 已判 P0 的多处表述', sum(hits) >= 8, '%d/%d' % (sum(hits), len(probes)))

  G18-02 乘数有界⇒避免误差放大                       → 行 287
  G18-03 正则化压制小方差方向                        → 行 2067
  G18-04 谱衰减⇒能量集中                          → 行 1987,2073,2436
  G18-05 κ(C) 判有效秩                         → 行 2171
  G18-07 定义 4.16 按列减均值                     → 行 1851
  G18-08 主成分得分 V^T X~                      → 行 1923
  G18-08 D3 改写版 Z_r=V_r^T X~               → 行 1941
  G18-11 式 (4.15) 无分母                      → 行 1368
  G14-05 Weyl "远小于"                        → 行 964
  D3 新增 ΔW* 主要方向 = PCA 主方向                 → 行 2091
  D3 内部矛盾：L2091 断言"主要方向正是 PCA 主方向"，L2436 易错点却把
    "ΔW 的主要能量集中在前 r 个奇异值方向，因此 LoRA 有效"列为错误说法。
  10 条探针中 10 条仍能在整合稿中检索到（不据此判断责任先后，只记录当前版本状态）。
  [通过] D3 整合稿当前版本仍保留 B1 已判 P0 的多处表述 | 10/10


## 单元 10 · D10：B1 关键数字抽样复核（哪些结论本组确认成立）

对 B1 报告中用于支撑 P0/P1 判定的解析式与数字逐项独立重算。

In [11]:
# (1) 预条件后条件数的解析式
n = 128; cc = np.cos(np.pi/(n+1)); kap_T = (3+2*cc)/(3-2*cc)
T = 3*np.eye(n) - np.eye(n,k=1) - np.eye(n,k=-1)
print('(1) cond(T/3) 解析式 = %.6f（B1/A2：4.99644）' % kap_T)
check('B1 的 4.99644 解析式正确', abs(kap_T-4.99644) < 1e-4)
# (2) LoRA 参数量
print('(2) r(d+k) = 8*(256+128) = %d，占全量 %d 的 %.4f%%（B1：9.375%%）'
      % (8*(256+128), 256*128, 100*8*(256+128)/(256*128)))
check('LoRA 参数量占比 9.375% 正确', abs(8*384/32768 - 0.09375) < 1e-12)
# (3) 三对角测例：右端为特征向量 1 步，换一般右端需更多步
def tri(nn):
    return 2*np.eye(nn) - np.eye(nn,k=1) - np.eye(nn,k=-1)
for nn in [8, 16, 32, 64]:
    Tn = tri(nn); xs = np.sin(np.pi*np.arange(1,nn+1)/(nn+1))
    _, k_eig, _, _ = cg(Tn, Tn@xs, tol=1e-10)
    _, k_gen, _, _ = cg(Tn, Tn@np.ones(nn), tol=1e-10)
    res = np.linalg.norm(Tn@xs - (xs@Tn@xs)/(xs@xs)*xs)/np.linalg.norm(xs)
    print('(3) n=%2d：特征残差 %.2e，RHS=T·sin 需 %d 步，RHS=T·1 需 %d 步（B1：n/2=%d）'
          % (nn, res, k_eig, k_gen, nn//2))
    check('n=%d 测例退化被复现' % nn, k_eig <= 2 and k_gen >= nn//2 - 2)
# (4) PCA 未中心化演示夹角
X6 = np.array([[1,1,1,1],[2,3,2,3],[3,2,4,2],[4,4,3,5],[5,6,5,6],[6,5,7,5]], float)
vc = np.linalg.svd(X6-X6.mean(axis=0), full_matrices=False)[2][0]
vu = np.linalg.svd(X6, full_matrices=False)[2][0]
ang = np.degrees(np.arccos(min(1, abs(vc@vu))))
X7 = np.array([[10.,0.],[10.,0.],[10.,3.],[10.,3.]])
v7c = np.linalg.svd(X7-X7.mean(axis=0), full_matrices=False)[2][0]
v7u = np.linalg.svd(X7, full_matrices=False)[2][0]
ang7 = np.degrees(np.arccos(min(1, abs(v7c@v7u))))
print('(4) A1 的 6×4 数据夹角 %.3f°（B1：1.296°）；对照数据 %.2f°（B1：81.3°）' % (ang, ang7))
check('B1 的"演示不可辨识"判定成立', ang < 3 and ang7 > 45)

(1) cond(T/3) 解析式 = 4.996444（B1/A2：4.99644）
  [通过] B1 的 4.99644 解析式正确
(2) r(d+k) = 8*(256+128) = 3072，占全量 32768 的 9.3750%（B1：9.375%）
  [通过] LoRA 参数量占比 9.375% 正确
(3) n= 8：特征残差 1.20e-16，RHS=T·sin 需 1 步，RHS=T·1 需 4 步（B1：n/2=4）
  [通过] n=8 测例退化被复现
(3) n=16：特征残差 3.43e-16，RHS=T·sin 需 1 步，RHS=T·1 需 8 步（B1：n/2=8）
  [通过] n=16 测例退化被复现
(3) n=32：特征残差 2.14e-16，RHS=T·sin 需 1 步，RHS=T·1 需 16 步（B1：n/2=16）
  [通过] n=32 测例退化被复现
(3) n=64：特征残差 2.98e-16，RHS=T·sin 需 2 步，RHS=T·1 需 32 步（B1：n/2=32）
  [通过] n=64 测例退化被复现
(4) A1 的 6×4 数据夹角 1.296°（B1：1.296°）；对照数据 81.28°（B1：81.3°）
  [通过] B1 的"演示不可辨识"判定成立


## 单元 11 · 汇总

按证据强度汇总本组判定；详细推导见《C1组-手动推导.md》，交叉核验见《C1组-批判报告.md》。

In [12]:
print('本 Notebook 共执行 %d 项本组检查：通过 %d，未通过 %d'
      % (len(CHECKS), sum(1 for _,o,_ in CHECKS if o), sum(1 for _,o,_ in CHECKS if not o)))
for name, ok, det in CHECKS:
    if not ok: print('  ✗', name, det)
print()
print('核心结论：')
print(' 1) B1 的 G18-05 反例方向错误：κ 不大 ⇒ 能量不集中，恰好支持"大 κ 是必要前提"；')
print('    驳充分性须用"κ 大但有效秩高"的反例（单元 1）。')
print(' 2) D3 整合稿当前版本仍保留 B1 已判 P0 的 4.4.4 三条因果链、定义 4.16、乘数有界表述；')
print('    且新增"ΔW* 主要方向正是 C 的 PCA 主方向"，被 A2 自己的反例推翻（单元 2、7、9）。')
print(' 3) B1 未推导的机理：Weyl 一阶项 u^T E v、残差界、(4.15) 严格形式、g_i 结构性约束（单元 3–6）。')
print(' 4) B1 的 71 项检查数量属实，但含恒真/静态断言，须按证据强度分层（单元 8）。')
print(' 5) B1 的数值结论（4.99644、9.375%、三对角退化、PCA 夹角、增长因子）本组复核成立（单元 10）。')

本 Notebook 共执行 31 项本组检查：通过 31，未通过 0

核心结论：
 1) B1 的 G18-05 反例方向错误：κ 不大 ⇒ 能量不集中，恰好支持"大 κ 是必要前提"；
    驳充分性须用"κ 大但有效秩高"的反例（单元 1）。
 2) D3 整合稿当前版本仍保留 B1 已判 P0 的 4.4.4 三条因果链、定义 4.16、乘数有界表述；
    且新增"ΔW* 主要方向正是 C 的 PCA 主方向"，被 A2 自己的反例推翻（单元 2、7、9）。
 3) B1 未推导的机理：Weyl 一阶项 u^T E v、残差界、(4.15) 严格形式、g_i 结构性约束（单元 3–6）。
 4) B1 的 71 项检查数量属实，但含恒真/静态断言，须按证据强度分层（单元 8）。
 5) B1 的数值结论（4.99644、9.375%、三对角退化、PCA 夹角、增长因子）本组复核成立（单元 10）。
